## Metadata EDA

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, ArrayType, MapType

# Initialize Spark session
spark = SparkSession.builder.appName("Metadata EDA on Spark").getOrCreate()

# Read lines from jsonl file\
lines = spark.read.text("meta_Video_Games.jsonl")

# Define the Schema of the Metadata Fields
schema = StructType([
    StructField("main_category", StringType(), True),
    StructField("title", StringType(), True),
    StructField("average_rating", FloatType(), True),
    StructField("rating_number", IntegerType(), True),
    StructField("features", StringType(), True),
    StructField("description", StringType(), True),
    StructField("price", FloatType(), True),
    StructField("images", ArrayType(StringType()), True),
    StructField("videos", ArrayType(StringType()), True),
    StructField("store", StringType(), True),
    StructField("categories", ArrayType(StringType()), True),
    StructField("details", StringType(), True),
    # StructField("details", MapType(StringType(), StringType()), True),
    StructField("parent_asin", StringType(), True),
    StructField("bought_together", ArrayType(StringType()), True)
])

# Parse each line of the file as a JSON object
json_df = lines.select(from_json(col("value"), schema).alias("data"))

# Flatten the nested struct into separate columns
meta_df = json_df.select("data.*")

# Show the DataFrame
meta_df.show()

# Print the schema to inspect the data structure
meta_df.printSchema()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/30 22:00:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------------------+--------------------+--------------+-------------+---------------------+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+---------------+
|       main_category|               title|average_rating|rating_number|             features|         description| price|              images|              videos|               store|          categories|             details|parent_asin|bought_together|
+--------------------+--------------------+--------------+-------------+---------------------+--------------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+---------------+
|         Video Games|Dash 8-300 Profes...|           5.0|            1| ["Features Dash 8...|["The Dash 8-300 ...|  NULL|[{"thumb":"https:...|                  []|            Aerosoft|[Video Games, PC,...|{"Pricing":"The s...| B000

In [2]:
# Get the total number of rows in the DataFrame
total_rows = meta_df.count()

# Print the total number of rows
print(f"Total number of rows: {total_rows}")

# Brief Description of the DataFrame
meta_df.describe().show()

Total number of rows: 137269


25/03/30 19:36:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------------+--------------------+------------------+------------------+--------------------+--------------------+-----------------+----------------+--------------------+--------------------+
|summary|main_category|               title|    average_rating|     rating_number|            features|         description|            price|           store|             details|         parent_asin|
+-------+-------------+--------------------+------------------+------------------+--------------------+--------------------+-----------------+----------------+--------------------+--------------------+
|  count|       126234|              137269|            137269|            137269|              137269|              137269|            61972|          132908|              137269|              137269|
|   mean|         NULL|           7159.6875| 3.995296092671304| 244.3050870917687|                NULL|                NULL|45.71735187750276|             NaN|                NULL|3.1734018962

Based on a brief exploration, there are a total of **137,269** Entries in the Metadata. This means that there are a total of **137,269** items that have been put up on the Amazon Marketplace in the **Video Games** Section from May 1999 to Sept. 2023.

From the Description of the Dataframe, there are some fields such as `main_category` and `price` where the count does not match the number of entries. This is likely due to NULL values.

### Checking for NULL Values

In [ ]:
from pyspark.sql import functions as F

# Count the number of null values in each column
df_nulls = meta_df.select([
    F.count(F.when(
        F.col(c).isNull(), c)).alias(c)
              for c in meta_df.columns
              ])

# Show the result
df_nulls.show()

+-------------+-----+--------------+-------------+--------+-----------+-----+------+------+-----+----------+-------+-----------+---------------+
|main_category|title|average_rating|rating_number|features|description|price|images|videos|store|categories|details|parent_asin|bought_together|
+-------------+-----+--------------+-------------+--------+-----------+-----+------+------+-----+----------+-------+-----------+---------------+
|        11035|    0|             0|            0|       0|          0|75297|     0|     0| 4361|         0|      0|          0|         137269|
+-------------+-----+--------------+-------------+--------+-----------+-----+------+------+-----+----------+-------+-----------+---------------+



As shown, there are four fields with numerous NULL Values:

- **Main Category** : 11035 Null Values

- **Price** : 75297 Null Values

- **Store** : 4361 Null Values

- **Bought Together** : 137259 Null Values

As seen from above, the `bought_together` field is NULL for all entries in the Dataframe. This allows us to easily removed this field later onwards. However, to get a better understanding on the other 3 fields, we will check for duplicates first.

### Checking for Duplicates

In [4]:
# Get the total number of rows
total_rows = meta_df.count()

# Get the number of rows after dropping duplicates
unique_rows = meta_df.dropDuplicates().count()

# Calculate the number of duplicate rows
duplicate_rows = total_rows - unique_rows

# Print the number of duplicate rows
print(f"Total number of rows: {total_rows}")
print(f"Number of unique rows: {unique_rows}")
print(f"Number of duplicate rows: {duplicate_rows}")

Total number of rows: 137269
Number of unique rows: 137269
Number of duplicate rows: 0


After a quick check, there were **NO** Duplicates. Meaning that all the entries in the dataset is unique. Now we will check the 3 fields: `main_category`,`price` and `store` to better understand what we should do with the Null Values.

In [10]:
# Get unique values and collect them into a list
unique_values = meta_df.select("main_category").distinct().rdd.flatMap(lambda x: x).collect()

# Print the number of unique values
print(f"Number of unique values in 'main_category': {len(unique_values)}")

# Print the unique values
print("Unique values in 'main_category':")
print(unique_values)

Number of unique values in 'main_category': 37
Unique values in 'main_category':
['Audible Audiobooks', 'Computers', 'All Electronics', 'Home Audio & Theater', 'Pet Supplies', 'Toys & Games', 'Baby', 'Sports & Outdoors', 'Grocery', 'Video Games', 'Automotive', 'Books', 'Amazon Home', 'Industrial & Scientific', 'Health & Personal Care', 'Cell Phones & Accessories', 'Arts, Crafts & Sewing', 'Amazon Devices', 'Digital Music', 'Software', 'Tools & Home Improvement', 'Movies & TV', 'All Beauty', 'Office Products', 'Camera & Photo', 'Buy a Kindle', 'Musical Instruments', 'Portable Audio & Accessories', 'AMAZON FASHION', 'GPS & Navigation', 'Car Electronics', 'Appliances', 'Gift Cards', 'Collectible Coins', 'Handmade', '', None]


As shown, there are a total of 37 unique main categories, 2 of them being Blank and None Values, therefore a total of 35 main categories.

The team felt that 35 categories might be too many. It could make the analysis more complex and harder to work with. To simplify things, we decided to reduce the number of categories to 10. This change will help make the data easier to handle and understand.

The goal of reducing the categories is to:

1. Make the data easier to use by having fewer categories.

2. Make the results clearer and easier to explain.

3. Focus on the most important categories that matter for the analysis.

Here is the final 10 **Main Categories** that the team used:

### **1. Video Games**

### **2. Computers**

### **3. All Electronics**

### **4. Toys  & Games**
- Which includes: `Toys & Games`, `Baby`, `Musical Instruments`

### **5. Software**
- Which includes: `Software`, `Collectible Coins`

### **6. Meda**
- Which includes: `Buy a Kindle`, `Movies & TV`, `Books`, `Digital Music`, `Audible Aduiobook`

### **7. Cell Phone & Camera w. Accessories**
- Which includes: `Cell Phones & Accessories`, `Camera & Photo`, `Portable Audio & Accessories`

### **8. Sports & Health**
- Which includes: `Sports & Outdoors`, `All Beauty`, `Health & Personal Care`, `Pet Supplies`

### **9.Daily Gadgets**
- Which includes: `Industrial & Scientific`, `Amazon Home`, `Home Audio & Theater`, `Tools & Home Improvement`, `Office Products`, `Automotive`, `Car Electronics`, `GPS & Navigation`, `Appliances`, `Amazon Devices`

### **10. Others**
- Which includes the rest: `Others`, `Handmade`, `Gift Cards`, `Grocery`, `AMAZON FASHION`, `""`, `NULL`

## User Reviews EDA

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, ArrayType, MapType

# Initialize Spark session
spark = SparkSession.builder.appName("User Review EDA on Spark").getOrCreate()

25/04/03 13:44:17 WARN Utils: Your hostname, AA.local resolves to a loopback address: 127.0.0.1; using 172.20.10.6 instead (on interface en0)
25/04/03 13:44:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/03 13:44:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
reviews_df = spark.read.json("../Video_Games.jsonl")

reviews_df.show(10, False)

+----------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:
# Print the schema to inspect the data structure
reviews_df.printSchema()

root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [ ]:
# Get the total number of rows
total_rows = reviews_df.count()

# Get the number of rows after dropping duplicates
unique_rows = reviews_df.dropDuplicates().count()

# Calculate the number of duplicate rows
duplicate_rows = total_rows - unique_rows

# Print the number of duplicate rows
print(f"Total number of rows: {total_rows}")
print(f"Number of unique rows: {unique_rows}")
print(f"Number of duplicate rows: {duplicate_rows}")

In [6]:
# Get duplicate rows by checking for exact matches across all columns
duplicate_rows_df = reviews_df.join(reviews_df, on=[col for col in reviews_df.columns], how='inner')

# Show the duplicate rows
duplicate_rows_df.show(10, False)

+----------+------------+------+-----------+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+----------------------------------------------------------------------+----------------------------+-----------------+
|asin      |helpful_vote|images|parent_asin|rating|text                                                                                         

In [ ]:
from pyspark.sql.functions import from_unixtime, date_format, col

# Assuming 'timestamp_col' is your column with Unix timestamps in milliseconds
formatted_df = duplicate_rows_df.withColumn(
    "formatted_timestamp",
    date_format(from_unixtime(col("timestamp") / 1000), "yyyy-MM-dd-HH-mm-ss")
)

# Show the result
formatted_df.show()


+----------+------------+------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+-------------------+
|      asin|helpful_vote|images|parent_asin|rating|                text|    timestamp|               title|             user_id|verified_purchase|formatted_timestamp|
+----------+------------+------+-----------+------+--------------------+-------------+--------------------+--------------------+-----------------+-------------------+
|0143130838|           0|    []| 0143130838|   3.0|Actually - it's p...|1473651647000|         Three Stars|AH3GXRJKD2IDZJD6H...|            false|2016-09-12-11-40-47|
|0143130838|           0|    []| 0143130838|   5.0|A wonderful book!...|1474852876000|         A Must Read|AH2A62CN5J2VWWMKQ...|             true|2016-09-26-09-21-16|
|0143130838|           0|    []| 0143130838|   5.0|I bought it to gi...|1471233189000|I bought it to gi...|AGP7YSRZENBT6VKYS...|             true|2016-08-15-11-53-09

25/03/31 12:31:21 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 922421 ms exceeds timeout 120000 ms
25/03/31 12:31:22 WARN SparkContext: Killing executors is not supported by current scheduler.
25/03/31 12:47:01 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [ ]:
from pyspark.sql import functions as F

# Count the number of null values in each column
df_nulls = reviews_df.select([
    F.count(F.when(
        F.col(c).isNull(), c)).alias(c)
              for c in reviews_df.columns
              ])

# Show the result
df_nulls.show()